# 20 — Block 2: Proposal nhiều vòng (không người duyệt)

Chế độ mô phỏng của P-026 (CONTRACTS §12.4): `router fixed_budget` → `propose` → `judge accept_all` → `feedback keep_drop` → lặp tới khi đủ K.
Recipe khai báo ở `configs/proposals.yaml` theo định dạng `RecipeConfig` của P-026, nên chép qua lại được nguyên văn.

**Input:** output notebook 10. **GPU:** nên có (mã hoá text, field còn thiếu). **Output:** `art/proposals/<tên>/s<seed>_k<K>/`.

**Dùng kết quả của P-026 hoặc nơi khác:** đặt `export.csv` (cột `unit_id` dạng P-026 hoặc `image`) vào `art/proposals/<tên>/s<seed>_k<K>/`
của một Kaggle Dataset rồi gắn vào notebook 30. Không cần chạy notebook này.

**Hết giờ session / chạy tiếp:** Save Version → gắn output version vừa xong làm input của chính notebook này → chạy lại.
Mọi bước đã có kết quả được bỏ qua (cache).

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "missing"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
# luôn lấy bản mới nhất của COMMIT từ GitHub (kể cả khi /tmp/repo còn từ lần chạy trước trong session tương tác)
subprocess.run(["git", "-C", R, "fetch", "-q", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", "-f", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
for p in (f"{R}/src", R):   # `topicx` (harness) và `src.rav` (pipeline proposal, cùng đường import với P-026)
    if p not in sys.path:
        sys.path.insert(0, p)
for m in [m for m in sys.modules if m.split(".")[0] in ("topicx", "src")]:
    del sys.modules[m]   # chạy lại cell thì nạp lại code mới
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P, proposals as PR
cfg = C.load_config(SMOKE, SCENARIO)
commit = subprocess.run(["git", "-C", R, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code:", commit)
print(f"kịch bản={SCENARIO} | artifact={cfg['art']} | copy từ input: {C.sync_inputs(cfg)} file")

In [ ]:
STAGE = "proposal"
# Test (CPU, vài giây): protocol của harness + test lõi P-026 + component mới. Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
pcfg = PR.load_proposal_config(cfg)
RECIPES = list(pcfg["recipes"])   # bớt / thêm tên ở đây; định nghĩa trong configs/proposals.yaml
SEEDS, KS, B = cfg["seeds"], pcfg["ks"], pcfg["batch_size"]
print(RECIPES, "| seeds", SEEDS, "| K", KS, "| B", B)

## Chọn câu topic và τ trên dev (nhãn dev công khai; không đụng pool hay final test)

In [ ]:
choice = PR.choose_texts_and_tau(cfg, pcfg)
print("câu:", choice["texts"], "| τ:", choice["tau"], "| rule:", choice["tau_rule"])
pd.Series(choice["dev_ap"]).sort_values()

## Chạy từng recipe × seed × K (trong guard chặn đọc nhãn)

In [ ]:
for name in RECIPES:
    recipe = PR.resolve_recipe(pcfg["recipes"][name], choice["texts"], choice["tau"])
    for seed in SEEDS:
        for k in KS:
            PR.run_recipe(cfg, name, recipe, k, B, seed, goal=pcfg["goal"])
ex = PR.list_exports(cfg)
display(ex.drop(columns="path"))

In [ ]:
# Mỗi vòng của một lần chạy: số frame, điểm gộp trung bình, điểm từng tiêu chí (breakdown của P-026)
import json
rows = []
for r in ex.itertuples():
    for line in (r.path.parent / "history.jsonl").read_text(encoding="utf-8").splitlines():
        h = json.loads(line)
        if h["type"] == "proposal":
            rows.append({"name": r.name, "seed": r.seed, "k": r.k, "round": h["n"], "n": len(h["picks"]),
                         "combined_mean": np.mean([p["combined"] for p in h["picks"]]) if h["picks"] else np.nan})
display(pd.DataFrame(rows).groupby(["name", "k"]).agg(rounds=("round", "max"), frames=("n", "sum"), combined_mean=("combined_mean", "mean")))

## Gói kết quả
Zip file json/csv/md nhỏ (không có weights, embedding, nhãn oracle) → tab Output.

In [ ]:
P.bundle(cfg, STAGE)